In [ ]:
!rm -rf /content/drive/MyDrive/topk-eval/B3_LLM_qwen3-8b

## Cell 1 — Install

In [ ]:
%pip -q uninstall -y transformers
%pip -q install --no-cache-dir \
  "transformers>=4.51.0" \
  "sentence-transformers>=2.7.0" \
  "accelerate" \
  "bitsandbytes>=0.43.0" \
  "huggingface_hub" \
  "datasets" \
  "numpy<2" \
  "faiss-cpu" \
  "scikit-learn"
print("Library terpasang. RESTART runtime sebelum lanjut.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 292.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 281.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 240.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 237.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 187.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requi

> 🔁 RESTART runtime.

## Cell 2 — Imports + GPU

In [ ]:
import random, os, json, time, gc, warnings, re
import numpy as np, pandas as pd
import torch, faiss
import torch.nn.functional as F
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    tot = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM {tot:.1f} GB")
print("device:", device, "| seed:", SEED)

GPU: Tesla T4 | VRAM 15.6 GB
device: cuda | seed: 42


## Cell 3 — KONFIGURASI LITE

In [ ]:
# ============ KONFIGURASI LITE ============
PAIR_ID          = "B3_LLM_qwen3-8b"
STRATEGY         = "B"
KATEGORI         = "LLM (8B-8B, all-Qwen3, scale-reduced)"
LM_NAME          = "Qwen/Qwen3-8B"
EMBED_MODEL_NAME = "Qwen/Qwen3-Embedding-8B"
EMB_FAMILY       = "qwen3"

DATASET_NAME = "rajpurkar/squad_v2"
SPLIT        = "validation"

K_MIN, K_MAX     = 1, 10
N_PILOT          = 10           # turun dari 18
N_FULL           = 40           # TURUN dari 100 (20 ans + 20 unans)
CORPUS_LIMIT     = 500          # TURUN dari 1204 (all)
EMB_BATCH        = 2
GEN_BATCH        = 1
MAX_NEW_TOKENS   = 100          # turun dari 150
MAX_INPUT_TOKENS = 2048

USE_4BIT_LM    = True
USE_4BIT_EMBED = True

SAVE_DIR = f"/content/drive/MyDrive/topk-eval/{PAIR_ID}"
print(f"PAIR : {PAIR_ID} (LITE: N_FULL={N_FULL}, CORPUS={CORPUS_LIMIT})")
print(f"LM   : {LM_NAME} 4-bit")
print(f"EMBED: {EMBED_MODEL_NAME} 4-bit")

PAIR : B3_LLM_qwen3-8b (LITE: N_FULL=40, CORPUS=500)
LM   : Qwen/Qwen3-8B 4-bit
EMBED: Qwen/Qwen3-Embedding-8B 4-bit


## Cell 4 — Dataset (40 samples)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(SAVE_DIR, exist_ok=True)

from datasets import load_dataset
ds = load_dataset(DATASET_NAME, split=SPLIT).remove_columns(["title"])

def clean_minimal(t):
    t = t.replace("\n", " ").replace("\t", " ")
    return re.sub(r"\s+", " ", t).strip()

contexts_clean = [clean_minimal(c) for c in ds["context"]]
passages_all = list(dict.fromkeys(contexts_clean))
print("QA pairs:", len(ds), "| passages unik:", len(passages_all))

ans_idx   = [i for i in range(len(ds)) if len(ds["answers"][i]["text"]) > 0]
unans_idx = [i for i in range(len(ds)) if len(ds["answers"][i]["text"]) == 0]
rng = random.Random(SEED)
rng.shuffle(ans_idx); rng.shuffle(unans_idx)
n_each = N_FULL // 2
mixed = []
for a, u in zip(ans_idx[:n_each], unans_idx[:n_each]):
    mixed.extend([a, u])
rows = []
for i in mixed:
    texts = ds["answers"][i]["text"]
    rows.append({"qi": i, "question": ds["question"][i],
                 "gold_answer": texts[0] if len(texts) > 0 else "",
                 "gold_is_unanswerable": len(texts) == 0})
df_eval = pd.DataFrame(rows).reset_index(drop=True)
print(f"Eval LITE: {len(df_eval)} (ans={(~df_eval['gold_is_unanswerable']).sum()}, "
      f"unans={df_eval['gold_is_unanswerable'].sum()})")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
QA pairs: 11873 | passages unik: 1204
Eval LITE: 40 (ans=20, unans=20)


## Cell 5 — PHASE 1 LITE dgn checkpoint setiap 100 passages

In [ ]:
# ===== PHASE 1 LITE — checkpoint setiap 100 passages =====
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig

art_p_emb     = os.path.join(SAVE_DIR, "p_emb.npy")
art_passages  = os.path.join(SAVE_DIR, "passages.json")
art_index     = os.path.join(SAVE_DIR, "faiss.index")
art_retr      = os.path.join(SAVE_DIR, f"retrieval_top{K_MAX}.json")
art_checkpoint = os.path.join(SAVE_DIR, "p_emb_partial.npy")

passages = passages_all[:CORPUS_LIMIT]

# ---- decide: load atau compute? ----
need_compute = True
if all(os.path.exists(p) for p in [art_passages, art_index, art_retr]):
    pas = json.load(open(art_passages))
    ret = json.load(open(art_retr))["idx"]
    if len(pas) == len(passages) and len(ret) == len(df_eval):
        print("✅ Phase 1 LENGKAP -> LOAD")
        passages = pas
        index = faiss.read_index(art_index)
        retrieved_idx = ret
        need_compute = False
        print(f"   {len(passages)} passages | retrieved untuk {len(retrieved_idx)} queries")
    else:
        print(f"⚠️  artefak ukurannya beda (config={len(passages)},{len(df_eval)}; "
              f"artefak={len(pas)},{len(ret)}) -> hapus & compute baru")
        for p in [art_p_emb, art_passages, art_index, art_retr, art_checkpoint]:
            if os.path.exists(p): os.remove(p)
else:
    print("⚙️  artefak belum lengkap -> compute baru")

if need_compute:
    print(f"   {len(passages)} passages × Qwen3-Embedding-8B 4-bit, ~5 menit")

    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
    ) if USE_4BIT_EMBED else None

    emb_tok = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME, trust_remote_code=True,
                                            padding_side="left")
    if emb_tok.pad_token is None: emb_tok.pad_token = emb_tok.eos_token

    emb_kwargs = dict(trust_remote_code=True, device_map="auto")
    if bnb is not None: emb_kwargs["quantization_config"] = bnb
    else: emb_kwargs["torch_dtype"] = torch.float16
    emb_mdl = AutoModel.from_pretrained(EMBED_MODEL_NAME, **emb_kwargs).eval()
    print(f"   VRAM bebas setelah load: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

    @torch.no_grad()
    def _encode(texts, is_query=False):
        if is_query:
            instr = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: "
            texts = [instr + t for t in texts]
        all_embs = []
        for i in range(0, len(texts), EMB_BATCH):
            batch = texts[i:i+EMB_BATCH]
            enc = emb_tok(batch, padding=True, truncation=True, max_length=1024,
                          return_tensors="pt").to(emb_mdl.device)
            out = emb_mdl(**enc)
            embs = F.normalize(out.last_hidden_state[:, -1, :].float(), p=2, dim=-1)
            all_embs.append(embs.cpu().numpy().astype(np.float32))
        return np.concatenate(all_embs, axis=0)

    # checkpoint resume
    p_emb_acc = None
    start_idx = 0
    if os.path.exists(art_checkpoint):
        p_emb_acc = np.load(art_checkpoint)
        start_idx = p_emb_acc.shape[0]
        print(f"🔄 RESUME checkpoint: {start_idx}/{len(passages)}")

    CHK_EVERY = 100
    t0 = time.time()
    for chunk_start in range(start_idx, len(passages), CHK_EVERY):
        chunk_end = min(chunk_start + CHK_EVERY, len(passages))
        chunk_emb = _encode(passages[chunk_start:chunk_end], is_query=False)
        p_emb_acc = chunk_emb if p_emb_acc is None else np.concatenate([p_emb_acc, chunk_emb], axis=0)
        np.save(art_checkpoint, p_emb_acc)
        elapsed = time.time() - t0
        rate = max((chunk_end - start_idx) / max(elapsed, 0.1), 0.01)
        eta = (len(passages) - chunk_end) / rate / 60
        print(f"   {chunk_end}/{len(passages)} | elapsed {elapsed/60:.1f}m | ETA {eta:.1f}m")

    p_emb = p_emb_acc
    print(f"✅ Passages encoded: {p_emb.shape}")

    index = faiss.IndexFlatIP(p_emb.shape[1])
    index.add(p_emb)

    print(f"Encoding {len(df_eval)} queries...")
    q_emb = _encode(df_eval["question"].tolist(), is_query=True)
    D, I = index.search(np.ascontiguousarray(q_emb), K_MAX)
    retrieved_idx = I.tolist()

    print(f"Sanity sampel 0: Q={df_eval['question'][0][:70]}")
    print(f"           Top-1={passages[retrieved_idx[0][0]][:100]}")

    np.save(art_p_emb, p_emb)
    faiss.write_index(index, art_index)
    json.dump(passages, open(art_passages, "w"))
    json.dump({"idx": retrieved_idx, "score": D.tolist()}, open(art_retr, "w"))
    if os.path.exists(art_checkpoint): os.remove(art_checkpoint)

    del emb_mdl, emb_tok, p_emb
    gc.collect(); torch.cuda.empty_cache()
    print("✅ Phase 1 selesai & disimpan.")

if torch.cuda.is_available():
    print(f"VRAM bebas: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

⚙️  artefak belum lengkap -> compute baru
   500 passages × Qwen3-Embedding-8B 4-bit, ~5 menit


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

   VRAM bebas setelah load: 10.6 GB
   100/500 | elapsed 0.5m | ETA 2.1m
   200/500 | elapsed 1.0m | ETA 1.5m
   300/500 | elapsed 1.6m | ETA 1.1m
   400/500 | elapsed 2.3m | ETA 0.6m
   500/500 | elapsed 3.0m | ETA 0.0m
✅ Passages encoded: (500, 4096)
Encoding 40 queries...
Sanity sampel 0: Q=Who did the geographic scholars work for? 
           Top-1=The first European to travel the length of the Amazon River was Francisco de Orellana in 1542. The B
✅ Phase 1 selesai & disimpan.
VRAM bebas: 15.4 GB


## Cell 6 — Helpers

In [ ]:
SEP = "\n\n---\n\n"
INSTRUCTION_CONFIG2 = """INSTRUCTION:
Answer the question ONLY based on the information in the CONTEXT. Do not guess or add facts not present in the CONTEXT.

If the CONTEXT does not contain the information needed to answer the question:
- Politely and specifically acknowledge the limitation (mention the aspect of the question that cannot be answered from the CONTEXT).
- Offer one relevant form of further assistance (clarify the question, or a related topic you can help with).
- Avoid generic or template responses.

OUTPUT FORMAT (REQUIRED):
Answer: [your answer based on the CONTEXT, or a polite and contextual abstain statement]
Evidence: [the part of the CONTEXT that supports the answer, or "none" if abstaining]
"""

def parse_output_soft(text):
    t = (text or "").strip()
    answer_text = t; evidence_text = ""
    m = re.search(r"(?ims)^\s*answer\s*:\s*(.+?)(?=\n\s*evidence\s*:|\Z)", t)
    if m: answer_text = m.group(1).strip()
    m = re.search(r"(?ims)^\s*evidence\s*:\s*(.+?)\Z", t)
    if m: evidence_text = m.group(1).strip()
    return None, answer_text, evidence_text, t

GEN_PARAMS = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                  temperature=0.0, top_p=1.0, use_cache=True)
print("Helpers Config 2 siap.")

Helpers Config 2 siap.


## Cell 7 — PHASE 2 dgn RESUME

In [ ]:
# ===== PHASE 2 dgn RESUME =====
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def load_lm():
    tok = AutoTokenizer.from_pretrained(LM_NAME, use_fast=True, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.pad_token_id = tok.eos_token_id
    tok.padding_side = "left"; tok.truncation_side = "left"

    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
    ) if USE_4BIT_LM else None

    if bnb:
        mdl = AutoModelForCausalLM.from_pretrained(
            LM_NAME, quantization_config=bnb, device_map="auto", trust_remote_code=True)
        print("  -> LM 4-bit nf4")
    else:
        mdl = AutoModelForCausalLM.from_pretrained(
            LM_NAME, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True)
    mdl.config.pad_token_id = tok.pad_token_id
    mdl.eval()
    return tok, mdl

def build_prompt(question, hits_passages, tok):
    context_text = SEP.join([f"CONTEXT {i+1}:\n{p}" for i, p in enumerate(hits_passages)])
    user_content = f"{INSTRUCTION_CONFIG2}\n\n{context_text}\n\nQUESTION:\n{question}\n"
    messages = [{"role": "user", "content": user_content}]
    try:
        return tok.apply_chat_template(messages, tokenize=False,
                                       add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate_batch(prompts, tok, mdl):
    enc = tok(prompts, return_tensors="pt", padding=True, truncation=True,
              max_length=MAX_INPUT_TOKENS).to(mdl.device)
    with torch.no_grad():
        out = mdl.generate(**enc, pad_token_id=tok.eos_token_id,
                           eos_token_id=tok.eos_token_id, **GEN_PARAMS)
    gen = out[:, enc["input_ids"].shape[1]:]
    return [t.strip() for t in tok.batch_decode(gen, skip_special_tokens=True)]

# === Build ctx_per_k & cek resume ===
ctx_per_k = {k: [None]*len(df_eval) for k in range(K_MIN, K_MAX+1)}
for k in range(K_MIN, K_MAX+1):
    for qi in range(len(df_eval)):
        idxs = retrieved_idx[qi][:k]
        ctx_per_k[k][qi] = [passages[j] for j in idxs if 0 <= j < len(passages)]

existing_gen_file = os.path.join(SAVE_DIR, "generations.json")
existing_gens = {}
if os.path.exists(existing_gen_file):
    try: existing_gens = json.load(open(existing_gen_file))
    except: pass

raw_outputs = {k: [None]*len(df_eval) for k in range(K_MIN, K_MAX+1)}
answers     = {k: [None]*len(df_eval) for k in range(K_MIN, K_MAX+1)}

for k_str, data in existing_gens.items():
    try:
        k = int(k_str)
        if K_MIN <= k <= K_MAX and "parsed" in data and len(data["parsed"]) == len(df_eval):
            raw_outputs[k] = data["raw"]
            answers[k]     = data["parsed"]
    except: pass

ks_done = [k for k in range(K_MIN, K_MAX+1) if answers[k][0] is not None]
ks_todo = [k for k in range(K_MIN, K_MAX+1) if answers[k][0] is None]

if not ks_todo:
    print(f"✅ Semua k sudah lengkap (resume). Skip Phase 2.")
else:
    print(f"ks done (resume): {ks_done}\nks todo: {ks_todo}")

    tok, mdl = load_lm()
    print(f"LM loaded | VRAM bebas: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

    # === SMOKE TEST + DISTRIBUSI ABSTAIN di 10 sampel pertama ===
    print("\n" + "="*60); print("SMOKE TEST + estimasi"); print("="*60)
    _tp = build_prompt(df_eval["question"][0], ctx_per_k[ks_todo[0]][0], tok)
    _t0 = time.time(); _out = generate_batch([_tp], tok, mdl)[0]; _dt = time.time() - _t0
    n_remaining = len(ks_todo) * len(df_eval)
    print(f"waktu 1 prompt: {_dt:.1f}s | estimasi sisa: {_dt*n_remaining/60:.1f} menit")
    assert _out.strip() != "", "❌ SMOKE TEST GAGAL"
    print(f"sample 0 RAW: {_out[:200]!r}")
    print("✅ Smoke test lulus, lanjut full loop\n")

    for k in ks_todo:
        t0 = time.time()
        prompts = [build_prompt(df_eval["question"][qi], ctx_per_k[k][qi], tok)
                   for qi in range(len(df_eval))]
        outs = []
        for i in range(0, len(prompts), GEN_BATCH):
            outs.extend(generate_batch(prompts[i:i+GEN_BATCH], tok, mdl))
        parsed = [parse_output_soft(o)[1] for o in outs]
        raw_outputs[k] = outs
        answers[k]     = parsed

        # cek abstain ratio
        abstain_kw = ["no information","cannot","unable to","not mentioned","context does not"]
        n_abst = sum(1 for p in parsed if any(kw in p.lower() for kw in abstain_kw))
        print(f"k={k:>2} | {time.time()-t0:5.1f}s | abstain_kw={n_abst}/{len(parsed)} | "
              f"sample[0]: {parsed[0][:60]!r}")

        # save incremental
        json.dump({str(kk): {"raw": raw_outputs[kk], "parsed": answers[kk]}
                   for kk in range(K_MIN, K_MAX+1) if answers[kk][0] is not None},
                  open(existing_gen_file, "w"), ensure_ascii=False)

    del mdl, tok; gc.collect(); torch.cuda.empty_cache()
    print("✅ PHASE 2 selesai.")

ks done (resume): []
ks todo: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  -> LM 4-bit nf4
LM loaded | VRAM bebas: 9.3 GB

SMOKE TEST + estimasi
waktu 1 prompt: 4.6s | estimasi sisa: 30.6 menit
sample 0 RAW: 'Answer: The CONTEXT does not provide information about which organization or entity the geographic scholars worked for.  \nEvidence: none'
✅ Smoke test lulus, lanjut full loop

k= 1 | 200.3s | abstain_kw=12/40 | sample[0]: 'The CONTEXT does not provide information about which organiz'
k= 2 | 227.1s | abstain_kw=10/40 | sample[0]: "The geographic scholars worked for the BBC's Unnatural Histo"
k= 3 | 237.0s | abstain_kw=8/40 | sample[0]: 'The question cannot be answered based on the provided CONTEX'
k= 4 | 257.6s | abstain_kw=6/40 | sample[0]: 'The question cannot be answered based on the provided CONTEX'
k= 5 | 269.5s | abstain_kw=8/40 | sample[0]: 'The question cannot be answered based on the provided CONTEX'
k= 6 | 306.3s | abstain_kw=5/40 | sample[0]: 'The question cannot be answered based on the provided CONTEX'
k= 7 | 310.9s | abstain_kw=7/40 | sam

## Cell 8 — PHASE 3 NLI scoring

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

nli_tok = AutoTokenizer.from_pretrained("MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli")
nli_mdl = AutoModelForSequenceClassification.from_pretrained(
    "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device).eval()
print("NLI ready")

def split_claims(text):
    if not text or not text.strip(): return []
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if len(s.strip()) > 3]

@torch.no_grad()
def nli(p, h):
    inp = nli_tok(p, h, truncation=True, max_length=512, return_tensors="pt").to(device)
    probs = torch.softmax(nli_mdl(**inp).logits, dim=-1)[0].cpu().float().numpy()
    return float(probs[0]), float(probs[2])

def faith_score(ctx, ans):
    if not ans or not ans.strip(): return 0.0
    claims = split_claims(ans)
    if not claims: return 0.0
    ctx = ctx[:3000]
    return sum(1 for c in claims if (e:=nli(ctx,c))[0] > 0.5 and e[0] > e[1]) / len(claims)

ABS_H = "This text is a refusal or acknowledgment that the question cannot be answered from the given context."
ANS_H = "This text provides a substantive factual answer to a question."

def abstain(text):
    if not text or not text.strip(): return "ABSTAIN"
    t = text[:1500]
    return "ABSTAIN" if nli(t, ABS_H)[0] > nli(t, ANS_H)[0] else "ANSWER"

def score_k(k, idxs):
    rows = []
    for qi in idxs:
        ctx = "\n\n".join(ctx_per_k[k][qi] or [])
        ans = answers[k][qi]
        gold_unans = bool(df_eval["gold_is_unanswerable"][qi])
        rows.append({"k":k,"gold_is_unanswerable":gold_unans,
                     "gt_label":"ABSTAIN" if gold_unans else "ANSWER",
                     "pred_label":abstain(ans),
                     "faithfulness":faith_score(ctx, ans),
                     "answer":ans,"gold_answer":df_eval["gold_answer"][qi]})
    return rows

def agg(per):
    df = pd.DataFrame(per); rows = []
    for k, g in df.groupby("k"):
        ans_mask = ~g["gold_is_unanswerable"]
        faith_ans = g.loc[ans_mask, "faithfulness"].mean() if ans_mask.any() else 0.0
        tp = int(((g["gt_label"]=="ABSTAIN")&(g["pred_label"]=="ABSTAIN")).sum())
        fp = int(((g["gt_label"]=="ANSWER")&(g["pred_label"]=="ABSTAIN")).sum())
        fn = int(((g["gt_label"]=="ABSTAIN")&(g["pred_label"]=="ANSWER")).sum())
        tn = int(((g["gt_label"]=="ANSWER")&(g["pred_label"]=="ANSWER")).sum())
        pr = tp/(tp+fp) if tp+fp>0 else 0.0
        rc = tp/(tp+fn) if tp+fn>0 else 0.0
        f1 = 2*pr*rc/(pr+rc) if pr+rc>0 else 0.0
        rows.append({"k":int(k),"faithfulness_answerable":round(faith_ans,4),
                     "abstention_precision":round(pr,4),"abstention_recall":round(rc,4),
                     "abstention_f1":round(f1,4),"tp":tp,"fp":fp,"fn":fn,"tn":tn,
                     "decision_score":round((faith_ans+f1)/2,4)})
    return df, pd.DataFrame(rows).set_index("k").sort_index()

pilot_idx = list(range(min(N_PILOT, len(df_eval))))
full_idx  = list(range(len(df_eval)))
print("\nScoring pilot...")
per_p = []
for k in range(K_MIN, K_MAX+1): per_p.extend(score_k(k, pilot_idx))
df18, summ18 = agg(per_p)
best18 = int(summ18["decision_score"].idxmax())
print(summ18.to_string()); print(f"BEST k (pilot) = {best18}")

print("\nScoring full...")
per_f = []
for k in range(K_MIN, K_MAX+1): per_f.extend(score_k(k, full_idx))
dffull, summfull = agg(per_f)
bestfull = int(summfull["decision_score"].idxmax())
print(summfull.to_string()); print(f"BEST k (full) = {bestfull}")

del nli_mdl, nli_tok; gc.collect(); torch.cuda.empty_cache()

config.json:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/395 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.65M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/18.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/870M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

NLI ready

Scoring pilot...
    faithfulness_answerable  abstention_precision  abstention_recall  abstention_f1  tp  fp  fn  tn  decision_score
k                                                                                                                  
1                    0.4000                0.5714                0.8         0.6667   4   3   1   2          0.5333
2                    0.3333                1.0000                0.6         0.7500   3   0   2   5          0.5417
3                    0.1000                0.6667                0.8         0.7273   4   2   1   3          0.4136
4                    0.0667                0.6667                0.8         0.7273   4   2   1   3          0.3970
5                    0.2000                0.5000                0.4         0.4444   2   2   3   3          0.3222
6                    0.1500                0.5000                0.4         0.4444   2   2   3   3          0.2972
7                    0.0667                0

## Cell 9 — Save

In [ ]:
FINAL_K = bestfull
df18.to_csv(os.path.join(SAVE_DIR, "topk_results_pilot.csv"), index=False)
dffull.to_csv(os.path.join(SAVE_DIR, "topk_results_full.csv"), index=False)
summ18.to_csv(os.path.join(SAVE_DIR, "topk_summary_pilot.csv"))
summfull.to_csv(os.path.join(SAVE_DIR, "topk_summary_full.csv"))

summary = {
    "pair_id": PAIR_ID, "lm": LM_NAME, "embed_model": EMBED_MODEL_NAME,
    "scale_note": f"LITE: N_FULL={N_FULL}, CORPUS_LIMIT={CORPUS_LIMIT} (smaller than other pairs due to T4 VRAM)",
    "final_k": FINAL_K, "best_k_pilot": best18, "best_k_full": bestfull,
    "n_pilot": len(pilot_idx), "n_full": len(full_idx),
    "scores_pilot": json.loads(summ18.reset_index().to_json(orient="records")),
    "scores_full":  json.loads(summfull.reset_index().to_json(orient="records")),
}
json.dump(summary, open(os.path.join(SAVE_DIR, "topk_validation_summary.json"), "w"), indent=2)
print(f"\n{'='*60}")
print(f"PASANGAN: {PAIR_ID} (LITE)")
print(f"TOP-K FINAL = {FINAL_K} (pilot k={best18}, penuh k={bestfull})")
print(f"{'='*60}")


PASANGAN: B3_LLM_qwen3-8b (LITE)
TOP-K FINAL = 1 (pilot k=2, penuh k=1)
